In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
ROOT = Path.cwd().parent.parent
sys.path.append(str(ROOT))
from src.loading_data.data_catalogue import DataCatalogue
from src.loading_data.load_data import get_clean_2022, get_master_2022_data
import pyreadstat
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from src.perturbation.create_perturbations import run_perturbation, create_and_join_diff_series
from perturbation.models.logistic import Logistic
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
def one_hot_encode_frame(df : pd.DataFrame):
    dc = DataCatalogue()
    discrete_vars = dc.get_perturbation_catogs()
    encoder = OneHotEncoder(drop='first', handle_unknown = 'ignore', sparse_output = False).set_output(transform = 'pandas')
    encoded = encoder.fit_transform(df[discrete_vars])
    df_encoded = pd.concat([df, encoded], axis = 1).drop(columns = discrete_vars)
    return df_encoded

In [3]:
df = get_clean_2022()
df_encoded = one_hot_encode_frame(df)

Loaded primary frame...
Loaded secondary frame...
Done.


In [4]:
print(df_encoded.isna().sum())

serial              0
year                0
LCA_Class           0
Age9                0
Educ6               0
NSSEC5            873
IMD10               0
MEMS7_ALL           0
Motiva_POP          0
motivd_POP          0
inclus_a         2983
inclus_b         3155
inclus_c         3082
anxious          7500
comm1            7501
comm2            7605
happy            7501
indev            7499
indevtry         7500
lone              227
worthw           7502
active              0
Gend3_2.0           0
Gend3_3.0           0
Eth7_2.0            0
Eth7_3.0            0
Eth7_4.0            0
Eth7_5.0            0
Eth7_6.0            0
Eth7_7.0            0
WorkStat8_1.0       0
WorkStat8_2.0       0
WorkStat8_3.0       0
WorkStat8_4.0       0
WorkStat8_5.0       0
WorkStat8_6.0       0
WorkStat8_7.0       0
HHLiv9_1.0          0
HHLiv9_2.0          0
HHLiv9_3.0          0
HHLiv9_4.0          0
HHLiv9_5.0          0
HHLiv9_6.0          0
HHLiv9_7.0          0
HHLiv9_8.0          0
dtype: int

In [5]:
df_encoded.columns
drop_cols = ['serial', 'year', 'LCA_Class', 'MEMS7_ALL']
keep_vars = [var for var in df_encoded.columns if var not in drop_cols]
labels = df_encoded['LCA_Class']
Y = df_encoded['active']
X = df_encoded[keep_vars]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.4, random_state = 42, stratify = labels)

In [6]:
test_set_clusters = df_encoded.loc[X_test.index, 'LCA_Class']
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
logistic = Logistic()
logistic.fit_logistic(X_train_scaled, Y_train)
perturbed_df = run_perturbation(X_test, logistic, scaler)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
coef = pd.Series(
    logistic.get_model().coef_[0],
    index=X_train.columns
)
print(coef.sort_values())

In [ ]:
perturbed_df['labels'] = test_set_clusters

In [ ]:
cluster_breakdowns = create_and_join_diff_series(perturbed_df)

In [ ]:
final = cluster_breakdowns.drop(columns='labels')

In [ ]:
plt.figure(figsize=(15, 10))
sns.heatmap(
    final,
    annot=True
)

In [8]:
df = get_master_2022_data()

In [9]:
df.isna().sum()

serial         0
year           0
LCA_Class      0
Age9           0
Gend3          0
Eth7           0
Educ6          0
NSSEC5       873
IMD10          0
WorkStat8      0
HHLiv9         0
dtype: int64